In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "local",
    **{
        "type": "sql",
        "uri": "sqlite:///warehouse/iceberg/catalog.db",
        "warehouse": "warehouse/iceberg",
    }
)

catalog.create_namespace_if_not_exists("school")


In [ ]:
import pyarrow as pa
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, IntegerType, StringType

schema = Schema(
    NestedField(1, "id", IntegerType(), required=True),
    NestedField(2, "name", StringType(), required=False),
)

table = catalog.create_table_if_not_exists(
    "school.students",
    schema=schema,
)

import pyarrow as pa

data = pa.table({"id": [1, 2, 3], "name": ["Alice", "Bob", "Carol"]})
table.append(data)

df = table.scan().to_arrow().to_pandas()
print(df)

for snap in table.snapshots():
    print(snap)